# StylePops — FashionCLIP LoRA Eğitimi (Colab GPU)

**Önce:** Runtime → **Change runtime type → T4 GPU**

Bu notebook **44K MIT** verisini HuggingFace'ten çeker — 2GB Drive upload gerekmez.

**Git durumu:** Scriptler push edildikten sonra `git clone` çalışır. Henüz push yoksa aşağıdaki hücredeki repo URL'sini güncelleyin veya önce Mac'ten commit+push yapın.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q torch torchvision transformers accelerate peft datasets lightgbm scikit-learn joblib pillow

In [ ]:
import os
from google.colab import userdata

# Hugging Face token — Colab Secrets: HF_TOKEN
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN yüklendi (Colab Secrets)')
except Exception:
    print('HF_TOKEN secret yok — hız sınırlı olabilir')
    print('Colab: 🔑 Secrets → HF_TOKEN = hf_...')

In [ ]:
REPO = 'https://github.com/valueisinvalid/StylePops_Modules.git'
PROJECT_ROOT = '/content/StylePops_Modules'

if not os.path.isdir(PROJECT_ROOT):
    !git clone {REPO} {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git pull

!cd {PROJECT_ROOT} && git log -1 --oneline
import sys
sys.path.insert(0, f'{PROJECT_ROOT}/scripts')
os.chdir(PROJECT_ROOT)

## LoRA fine-tune (44K, HuggingFace)

İlk çalıştırmada HF dataset ~2GB indirilir (önbelleğe alınır).

Hızlı test için: `--max-samples 3000`

In [ ]:
# Tam eğitim (~1-3 saat T4, 1 epoch)
!python scripts/colab_train_aesthetic.py --source hf --epochs 1 --batch-size 32 --zip

# Hızlı test:
# !python scripts/colab_train_aesthetic.py --source hf --max-samples 3000 --epochs 1 --batch-size 32 --zip

In [ ]:
from google.colab import files
import os
zip_path = 'outputs/aesthetic_models.zip'
if os.path.exists(zip_path):
    print('İndiriliyor:', zip_path)
    files.download(zip_path)
else:
    print('Zip bulunamadı — eğitim hücresini çalıştırın')

## Mac'te kullanım

1. `aesthetic_models.zip` indir
2. `outputs/fashionclip_lora/` klasörünü Mac'te `StylePops_Modules/outputs/` altına çıkar
3. Compatibility head (Mac, yerel 44K ile):
   ```bash
   python scripts/train_compatibility_head.py
   ```
4. Streamlit / öneri sistemi otomatik LoRA modelini yükler (`clip_mode: lora`)